In [1]:
# Prerequisite Code
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST(
    root = "data",
    train = True,
    download = True,
    transform = ToTensor()
)
test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = ToTensor()
)
train_dataloader = DataLoader(training_data, batch_size = 64)
test_dataloader = DataLoader(test_data, batch_size = 64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [2]:
# Hyperparameters are adjustable paramets for the purpose of training a model optimally.
# We define the following hyperparameters for training:
#    Number of Epochs: the number of times to iterate over the dataset
#    Batch Size: The number of samples propagated through the network before hyperparameters are updated
#    Learning Rate: how to update model parameters at each batch/epoch. Smaller values yield slow learning and higher values yield unpredictiable results

learning_rate = 1e-3
batch_size = 64
epochs = 5

In [3]:
# Optimization Loop
# Once we set our hyperparameters, we can train our optimization loop. Each loop is called an epoch.
# Each epoch consists of two main parts: the training loop and the test loop
loss_fn = nn.CrossEntropyLoss()

In [4]:
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

In [5]:
# Full Implementation
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X,y) in enumerate(dataloader):
        #Compute Prediction and Loss
        pred = model(X)
        loss = loss_fn(pred, y)

        #Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [13]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 0.752195  [   64/60000]
loss: 0.835705  [ 6464/60000]
loss: 0.612330  [12864/60000]
loss: 0.810028  [19264/60000]
loss: 0.724890  [25664/60000]
loss: 0.724374  [32064/60000]
loss: 0.803179  [38464/60000]
loss: 0.775598  [44864/60000]
loss: 0.786676  [51264/60000]
loss: 0.740107  [57664/60000]
Test Error: 
 Accuracy: 72.7%, Avg loss: 0.738534 

Epoch 2
-------------------------------
loss: 0.721143  [   64/60000]
loss: 0.810587  [ 6464/60000]
loss: 0.586273  [12864/60000]
loss: 0.789605  [19264/60000]
loss: 0.707640  [25664/60000]
loss: 0.704556  [32064/60000]
loss: 0.780967  [38464/60000]
loss: 0.761779  [44864/60000]
loss: 0.768273  [51264/60000]
loss: 0.721749  [57664/60000]
Test Error: 
 Accuracy: 73.7%, Avg loss: 0.719388 

Epoch 3
-------------------------------
loss: 0.693695  [   64/60000]
loss: 0.787512  [ 6464/60000]
loss: 0.563463  [12864/60000]
loss: 0.771980  [19264/60000]
loss: 0.692528  [25664/60000]
loss: 0.687829  [32064/600